# bioacoustic-embedding-dynamics (Colab)

CPU is enough. Runtime, Run all. About 5-10 minutes.

Runs BMZ BirdNET on the short real wav bundled with bacpipe (~1 min), then PCA, UMAP, trajectory, change-points, and HMM. Bins are 15 s, not 60 s.

## 1. Clone

In [ ]:
REPO = "https://github.com/ST-48-1240162/bioacoustic-embedding-dynamics.git"

%cd /content
!rm -rf bioacoustic-embedding-dynamics
!git clone --depth 1 {REPO}
%cd bioacoustic-embedding-dynamics

## 2. Install

In [ ]:
import sys
!{sys.executable} -m pip install -q -r docs/colab-requirements.txt
!{sys.executable} -m pip install -q -e .

## 3. BirdNET manifest (real short clip)

In [ ]:
import sys
sys.path.insert(0, "scripts")
from pathlib import Path
from colab_wav_source import bacpipe_test_wav, install_bacpipe_for_test_wav
from bioacoustic_embedding_dynamics.adapters import bmz_birdnet_to_manifest

!{sys.executable} -m pip install -q soundfile bioacoustics-model-zoo[birdnet]
install_bacpipe_for_test_wav()
wav = bacpipe_test_wav()
print("wav:", wav)
manifest = Path("data/demo_birdnet.jsonl")
bmz_birdnet_to_manifest([wav], manifest, batch_size=8, min_confidence=0.0)
print("manifest lines:", sum(1 for _ in manifest.open()))

!{sys.executable} -m bioacoustic_embedding_dynamics.cli --manifest data/demo_birdnet.jsonl --out reports --seed 42 --bin-s 15

## 4. Summary and figures

In [ ]:
import json
from pathlib import Path
from IPython.display import Image, display

summary = json.loads(Path("reports/summary.json").read_text())
print(json.dumps(summary, indent=2))

for name in [
    "pca_species.png", "umap_species.png", "trajectory_pca.png",
    "changepoints.png", "trajectory_changepoints.png", "hmm_regimes.png", "shuffle_null.png",
]:
    display(Image(filename=str(Path("reports") / name)))